In [1]:
import requests
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

C:\Users\vasan\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
print("Downloading international match results...")

url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
response = requests.get(url)

with open(RAW_DIR / "results.csv", "wb") as f:
    f.write(response.content)

df = pd.read_csv(RAW_DIR / "results.csv")
print(f"Total matches: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Total matches: 49477
Date range: 1872-11-30 to 2026-06-27


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
print("Downloading shootout data...")

url = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"
response = requests.get(url)

with open(RAW_DIR / "shootouts.csv", "wb") as f:
    f.write(response.content)

shootouts = pd.read_csv(RAW_DIR / "shootouts.csv")
print(f"Total shootouts: {len(shootouts)}")
shootouts.head()

Total shootouts: 678


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [4]:
print("Downloading goalscorer data...")

url = "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv"
response = requests.get(url)

with open(RAW_DIR / "goalscorers.csv", "wb") as f:
    f.write(response.content)

goalscorers = pd.read_csv(RAW_DIR / "goalscorers.csv")
print(f"Total goalscorer records: {len(goalscorers)}")
goalscorers.head()

Total goalscorer records: 47606


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False


In [9]:
import itertools

wc2026_groups = {
    "A": ["Mexico", "South Korea", "Czech Republic", "South Africa"],
    "B": ["Canada", "Bosnia and Herzegovina", "Switzerland", "Qatar"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

# tournament schedule - retraining happens after each matchday completes
tournament_schedule = {
    "group_matchday_1": {"start": "2026-06-11", "end": "2026-06-17", "retrain_after": True},
    "group_matchday_2": {"start": "2026-06-18", "end": "2026-06-23", "retrain_after": True},
    "group_matchday_3": {"start": "2026-06-24", "end": "2026-06-27", "retrain_after": True},
    "round_of_32":      {"start": "2026-06-28", "end": "2026-07-03", "retrain_after": True},
    "round_of_16":      {"start": "2026-07-04", "end": "2026-07-07", "retrain_after": True},
    "quarterfinals":    {"start": "2026-07-09", "end": "2026-07-11", "retrain_after": True},
    "semifinals":       {"start": "2026-07-14", "end": "2026-07-15", "retrain_after": True},
    "third_place":      {"start": "2026-07-18", "end": "2026-07-18", "retrain_after": False},
    "final":            {"start": "2026-07-19", "end": "2026-07-19", "retrain_after": False},
}

# training data cutoff
df_train = df[df["date"] < "2026-06-11"].copy()
train_path = RAW_DIR / "training_data.csv"
df_train.to_csv(train_path, index=False)

# generate all 72 group stage fixtures
schedule_rows = []
match_id = 1

# matchday assignment per group
# each group plays 3 rounds - pair 1,2,3 per combination order
matchday_pairs = {1: [(0,1),(2,3)], 2: [(0,2),(1,3)], 3: [(0,3),(1,2)]}

for group, teams in wc2026_groups.items():
    for matchday, pairs in matchday_pairs.items():
        for i, j in pairs:
            schedule_rows.append({
                "match_id":  f"WC2026_G{match_id:03d}",
                "stage":     "Group Stage",
                "matchday":  f"Matchday {matchday}",
                "retrain_after_matchday": matchday,  # retrain after this matchday completes
                "group":     group,
                "home_team": teams[i],
                "away_team": teams[j],
                "date":      None,
                "venue":     None,
            })
            match_id += 1

schedule_df = pd.DataFrame(schedule_rows)

# save tournament schedule reference
import json
schedule_ref_path = RAW_DIR / "tournament_schedule.json"
with open(schedule_ref_path, "w") as f:
    json.dump(tournament_schedule, f, indent=2)

# empty results file
results_df = pd.DataFrame(columns=[
    "match_id", "date", "group", "stage", "matchday",
    "home_team", "away_team",
    "home_score", "away_score", "outcome"
])

schedule_path = RAW_DIR / "wc2026_schedule.csv"
results_path  = RAW_DIR / "wc2026_results.csv"
schedule_df.to_csv(schedule_path, index=False)
results_df.to_csv(results_path, index=False)

print("=" * 55)
print("ALL FILES SAVED")
print("=" * 55)
print(f"training_data.csv          → {len(df_train)} matches (up to Jun 10 2026)")
print(f"wc2026_schedule.csv        → {len(schedule_df)} group stage fixtures")
print(f"wc2026_results.csv         → empty (fill after each match)")
print(f"tournament_schedule.json   → retraining schedule")
print()
print("Retraining schedule:")
for stage, info in tournament_schedule.items():
    retrain = "→ RETRAIN after" if info["retrain_after"] else "→ no retrain"
    print(f"  {stage:<25} {info['start']} to {info['end']}  {retrain}")
print()
print("Group stage matchday structure:")
for _, row in schedule_df.head(12).iterrows():
    print(f"  {row['match_id']}  Group {row['group']}  MD{row['matchday'][-1]}  "
          f"{row['home_team']} vs {row['away_team']}")
print(f"  ... and {len(schedule_df)-12} more matches")

ALL FILES SAVED
training_data.csv          → 49405 matches (up to Jun 10 2026)
wc2026_schedule.csv        → 72 group stage fixtures
wc2026_results.csv         → empty (fill after each match)
tournament_schedule.json   → retraining schedule

Retraining schedule:
  group_matchday_1          2026-06-11 to 2026-06-17  → RETRAIN after
  group_matchday_2          2026-06-18 to 2026-06-23  → RETRAIN after
  group_matchday_3          2026-06-24 to 2026-06-27  → RETRAIN after
  round_of_32               2026-06-28 to 2026-07-03  → RETRAIN after
  round_of_16               2026-07-04 to 2026-07-07  → RETRAIN after
  quarterfinals             2026-07-09 to 2026-07-11  → RETRAIN after
  semifinals                2026-07-14 to 2026-07-15  → RETRAIN after
  third_place               2026-07-18 to 2026-07-18  → no retrain
  final                     2026-07-19 to 2026-07-19  → no retrain

Group stage matchday structure:
  WC2026_G001  Group A  MD1  Mexico vs South Korea
  WC2026_G002  Group A  MD1  C